In [1]:
# ⚠️ до импорта pyarrow — CPU VM без AVX
import os
os.environ["ARROW_USER_SIMD_LEVEL"] = "NONE"

import json, random
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ds = load_dataset("scanpatch/pii-ner-corpus-synthetic-controlled")
df_train_full = ds["train"].to_pandas()
df_test = ds["test"].to_pandas()

print(f"train_full: {len(df_train_full)}")
print(f"test:       {len(df_test)}")
print(f"Колонки:    {list(df_train_full.columns)}")

/home/mluser/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


train_full: 4786
test:       532
Колонки:    ['text', 'entity_starts', 'entity_ends', 'entity_labels', 'entity_texts', 'source', 'noisy']


In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Группировка по text: одинаковые тексты не должны попасть в разные сплиты
groups = df_train_full["text"].fillna("")

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, va_idx = next(gss.split(df_train_full, groups=groups))

df_tr = df_train_full.iloc[tr_idx].reset_index(drop=True)
df_va = df_train_full.iloc[va_idx].reset_index(drop=True)

print(f"train: {len(df_tr)}")
print(f"val:   {len(df_va)}  ({len(df_va)/len(df_train_full)*100:.1f}% от train_full)")
print(f"test:  {len(df_test)}  (штатный hold-out, не трогаем)")
print(f"seed:  {SEED}, split: GroupShuffleSplit(test_size=0.15, groups=text)")

train: 4069
val:   717  (15.0% от train_full)
test:  532  (штатный hold-out, не трогаем)
seed:  42, split: GroupShuffleSplit(test_size=0.15, groups=text)


In [3]:
def texts(df):
    return set(df["text"].dropna())

tr_t, va_t, te_t = texts(df_tr), texts(df_va), texts(df_test)
print("Пересечение text train∩val: ", len(tr_t & va_t))
print("Пересечение text train∩test:", len(tr_t & te_t))
print("Пересечение text val∩test:  ", len(va_t & te_t))
assert len(tr_t & va_t) == 0, "Утечка train/val!"
assert len(tr_t & te_t) == 0, "Утечка train/test!"
assert len(va_t & te_t) == 0, "Утечка val/test!"
print("OK: одинаковых текстов между сплитами нет")

Пересечение text train∩val:  0
Пересечение text train∩test: 0
Пересечение text val∩test:   0
OK: одинаковых текстов между сплитами нет


In [4]:
# source — домен (банк, телеком и т.п.). Разные тексты одного домена — не утечка.
# Но проверим, что нет одинаковых (source, text) в разных сплитах.
key_tr = set(zip(df_tr["source"].fillna(""), df_tr["text"].fillna("")))
key_va = set(zip(df_va["source"].fillna(""), df_va["text"].fillna("")))
key_te = set(zip(df_test["source"].fillna(""), df_test["text"].fillna("")))

print("(source,text) train∩val: ", len(key_tr & key_va))
print("(source,text) train∩test:", len(key_tr & key_te))
print("(source,text) val∩test:  ", len(key_va & key_te))

# Пересечение самих source между сплитами (информационно)
print("\nsource train∩test:", sorted(set(df_tr["source"]) & set(df_test["source"])))

(source,text) train∩val:  0
(source,text) train∩test: 0
(source,text) val∩test:   0

source train∩test: ['controlled']


In [5]:
def label_counts(df):
    c = {}
    for labels in df["entity_labels"]:
        if labels is None:
            continue
        for l in labels:
            c[l] = c.get(l, 0) + 1
    return pd.Series(c, dtype=int)

c_tr = label_counts(df_tr)
c_va = label_counts(df_va)
c_te = label_counts(df_test)

comp = pd.DataFrame({
    "train": c_tr, "val": c_va, "test": c_te
}).fillna(0).astype(int)
comp["val_share_of_train"] = (comp["val"] / comp["train"].replace(0, np.nan)).round(3)
comp["test_share_of_train"] = (comp["test"] / comp["train"].replace(0, np.nan)).round(3)
comp = comp.sort_values("train", ascending=False)
print(comp)

# ожидаемая доля val ≈ 0.15/0.85 ≈ 0.176
print(f"\nЦелевая доля val от train_full: {len(df_va)/len(df_train_full):.3f}")
print(f"Средняя фактическая val/train по типам: {comp['val_share_of_train'].mean():.3f}")
print(f"Разброс val/train по типам: "
      f"min={comp['val_share_of_train'].min():.3f}, "
      f"max={comp['val_share_of_train'].max():.3f}")

                            train  val  test  val_share_of_train  \
ip                           3751  659   579               0.176   
name                         2365  378   300               0.160   
first_name                   2358  404   285               0.171   
date                         2112  421   241               0.199   
address_city                 1861  355   232               0.191   
last_name                    1650  268   203               0.162   
mobile_phone                 1564  289   195               0.185   
address                      1453  256   180               0.176   
address_street               1407  226   151               0.161   
document_number              1088  173   115               0.159   
email                        1053  182   114               0.173   
middle_name                   820  130   105               0.159   
organization                  770  137   116               0.178   
address_house                 620   87    63    

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 6.1 Словарь типов сущностей — только из train
label_vocab = sorted({
    l for labels in df_tr["entity_labels"] if labels is not None for l in labels
})
print(f"label_vocab ({len(label_vocab)}): {label_vocab}")

# Проверка: все ли метки val/test покрыты train-словарём
labels_va = {l for labels in df_va["entity_labels"] if labels is not None for l in labels}
labels_te = {l for labels in df_test["entity_labels"] if labels is not None for l in labels}
print("Метки val, отсутствующие в train:", labels_va - set(label_vocab))
print("Метки test, отсутствующие в train:", labels_te - set(label_vocab))

# 6.2 TF-IDF для baseline бинарной классификации документа (есть ПДн / нет)
df_tr["has_pii"] = df_tr["entity_labels"].apply(lambda x: int(x is not None and len(x) > 0))
y_tr = df_tr["has_pii"].values
print(f"\nhas_pii в train: {np.bincount(y_tr)}")

vectorizer = TfidfVectorizer(
    lowercase=True, max_features=5000, ngram_range=(1, 2), min_df=2
)
X_tr = vectorizer.fit_transform(df_tr["text"].fillna(""))
X_va = vectorizer.transform(df_va["text"].fillna(""))
X_te = vectorizer.transform(df_test["text"].fillna(""))
print(f"TF-IDF: train={X_tr.shape}, val={X_va.shape}, test={X_te.shape}")

# 6.3 Словарь токенов — только из train
count_vec = CountVectorizer(lowercase=True, min_df=2)
count_vec.fit(df_tr["text"].fillna(""))
token_vocab = count_vec.vocabulary_
print(f"token vocab size: {len(token_vocab)}")

label_vocab (27): ['address', 'address_apartment', 'address_building', 'address_city', 'address_country', 'address_district', 'address_geolocation', 'address_house', 'address_postal_code', 'address_region', 'address_street', 'date', 'document_number', 'email', 'first_name', 'ip', 'last_name', 'middle_name', 'military_individual_number', 'mobile_phone', 'name', 'name_initials', 'nickname', 'organization', 'snils', 'tin', 'vehicle_number']
Метки val, отсутствующие в train: set()
Метки test, отсутствующие в train: set()

has_pii в train: [ 409 3660]
TF-IDF: train=(4069, 5000), val=(717, 5000), test=(532, 5000)
token vocab size: 18463


In [8]:
# Число признаков одинаково — значит, используется один и тот же словарь TF-IDF
assert X_va.shape[1] == X_tr.shape[1] == X_te.shape[1]
print(f"OK: одинаковое число признаков TF-IDF: {X_tr.shape[1]}")

# Словарь токенов не расширялся на val/test
print(f"OK: token vocab фиксирован на train: {len(token_vocab)} токенов")

# Все метки val/test покрыты label_vocab
assert labels_va <= set(label_vocab)
assert labels_te <= set(label_vocab)
print("OK: все метки val/test присутствуют в label_vocab из train")

OK: одинаковое число признаков TF-IDF: 5000
OK: token vocab фиксирован на train: 18463 токенов
OK: все метки val/test присутствуют в label_vocab из train


In [9]:
import joblib

art = os.path.expanduser("~/mlproject/artifacts")
os.makedirs(art, exist_ok=True)

# 8.1 Индексы разделения (воспроизводимость)
with open(f"{art}/split_indices.json", "w") as f:
    json.dump({
        "seed": SEED,
        "method": "GroupShuffleSplit(test_size=0.15, groups=text)",
        "train": tr_idx.tolist(),
        "val": va_idx.tolist(),
        "test": "native test split (532 objects)",
    }, f)

# 8.2 Словарь меток
with open(f"{art}/label_vocab.json", "w") as f:
    json.dump(label_vocab, f, ensure_ascii=False)

# 8.3 Обученные преобразования
joblib.dump(vectorizer, f"{art}/tfidf_vectorizer.joblib")
joblib.dump(count_vec, f"{art}/count_vectorizer.joblib")

# 8.4 Сами сплиты в parquet (для следующих ЛР)
df_tr.to_parquet(f"{art}/train.parquet")
df_va.to_parquet(f"{art}/val.parquet")
df_test.to_parquet(f"{art}/test.parquet")

# 8.5 Сводка
summary = {
    "seed": SEED,
    "split_method": "GroupShuffleSplit by text",
    "sizes": {"train": len(df_tr), "val": len(df_va), "test": len(df_test)},
    "leakage_text": {
        "train_val": len(tr_t & va_t),
        "train_test": len(tr_t & te_t),
        "val_test": len(va_t & te_t),
    },
    "label_vocab_size": len(label_vocab),
    "tfidf_features": int(X_tr.shape[1]),
    "token_vocab_size": len(token_vocab),
}
with open(f"{art}/split_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\nАртефакты: {art}/")

{
  "seed": 42,
  "split_method": "GroupShuffleSplit by text",
  "sizes": {
    "train": 4069,
    "val": 717,
    "test": 532
  },
  "leakage_text": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "label_vocab_size": 27,
  "tfidf_features": 5000,
  "token_vocab_size": 18463
}

Артефакты: /home/mluser/mlproject/artifacts/
